# Meeting 14: exploring data with matrix products

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 14.

Matrix calculations help us compare features and points in a dataset. Principal component analysis also uses an eigendecomposition to find the directions along which the data varies most. This notebook asks them of the ratings that 610 people gave to movies. It ends with a claim of your own about the movies, with the evidence for it.

Do it **after meeting 14**, on your own, and upload it. It takes the questions of the meeting step by step, in NumPy.

## How this works

- **This notebook needs no GPU.** Leave its runtime as it is.
- **First** choose **Save a copy in Drive** in the **File** menu, to save your work, and run the cells in order, from the top. To run a cell, click it and press **Shift + Enter**.
- **Work on your own.** There is no group for this meeting: every student does the notebook and writes its graded cells.
- **Seven parts**, with code to run. Six exercises ask you to complete a line of code, read a result, or obtain a function from an AI assistant, marked <font color="#c00000"><b>Your turn</b></font>, each with a **check** whose message says what to look at.
- **Eight cells ask for your own words**: one marked <font color="#c00000"><b>Your prompt</b></font>, the prompt you give an AI assistant, and seven marked <font color="#c00000"><b>Your thinking</b></font>. **They are what this notebook is graded on.**
- At the end, upload the notebook to Canvas, as the last section says.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with your name.

In [ ]:
# Setup. Run this cell first.
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd

SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
MOVIES = SITE + "movies/"                         # the folder of the ratings data

np.set_printoptions(precision=3, suppress=True, linewidth=120)
pd.set_option("display.width", 120)

BLUE, ORANGE, PURPLE, GREY = "#2a78d6", "#eb6834", "#6a3fb5", "#888888"
DIVERGING = LinearSegmentedColormap.from_list("diverging", [BLUE, "#e9e9e9", ORANGE])   # blue below 0, grey at 0, orange above


def short(title):
    """A title as it is said: 'Matrix, The (1999)' becomes 'The Matrix'; the year and any other parenthesis are dropped."""
    title = re.sub(r"\s*\(\d{4}\)\s*$", "", title)
    title = re.sub(r"\s*\([^)]*\)", "", title).strip()
    match = re.match(r"^(.*), (The|A|An)$", title)
    return f"{match.group(2)} {match.group(1)}" if match else title


def year_of(title):
    """The year in the parentheses at the end of a title, or 0 when there is none."""
    match = re.search(r"\((\d{4})\)\s*$", title)
    return int(match.group(1)) if match else 0


def tidy(ax):
    """The look of the charts of this notebook: no frame on the top and the right, a light grid."""
    ax.grid(True, color="#e1e0d9", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)


print("Setup is done.")

## <font color="#1967d2"><b>Part 1: The table becomes a matrix</b></font>

The data is a set of movie ratings collected by the MovieLens service of the University of Minnesota: **100,836 ratings** of 9,724 movies by **610 people**, given between 1996 and 2018, from half a star to five stars. The movie catalog has 9,742 titles; 18 have tags but no ratings. The people were selected at random among those who had rated at least 20 movies; no demographic information is included. The course keeps a copy for research use, with [the dataset's license](https://ikoutis.github.io/cs301-labs/unit-2/movies/README.txt) and [source and citation](https://ikoutis.github.io/cs301-labs/unit-2/movies/SOURCES.md).

The file `ratings.csv` has one row for each rating: who, which movie, how many stars, and when. A second file, `movies.csv`, gives the title and the genres of each movie. The next cell reads both.

In [ ]:
ratings = pd.read_csv(MOVIES + "ratings.csv")
movies = pd.read_csv(MOVIES + "movies.csv")
print(f"ratings: {ratings.shape[0]:,} rows and {ratings.shape[1]} columns")
print(ratings.head(3))
print()
print(f"movies: {movies.shape[0]:,} rows and {movies.shape[1]} columns")
print(movies.head(3))
print()
print(f"{ratings.userId.nunique()} people, {ratings.movieId.nunique():,} movies with at least one rating, {len(ratings):,} ratings")

**A list of triples.** Each row of `ratings` is a row number (the person), a column number (the movie) and a value (the stars): the table is the **COO format** of meeting 13, a list of the nonzero entries of a matrix of people by movies. Most entries of that matrix are empty: 610 people times 9,742 movies is 5.9 million cells, and 100,836 of them hold a rating.

The next cell counts the ratings of each movie and of each person, and draws two pictures: the movies from the most rated to the least, and the stars given.

In [ ]:
ratings_per_movie = ratings.groupby("movieId").size()
ratings_per_person = ratings.groupby("userId").size()
stars = ratings.rating.value_counts().sort_index()

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
ranked = np.sort(ratings_per_movie.to_numpy())[::-1]
left.plot(np.arange(1, len(ranked) + 1), ranked, color=BLUE, linewidth=2)
left.set_xscale("log")
left.set_yscale("log")
left.axhline(20, color=GREY, linewidth=1, linestyle="--")
left.set_xlabel("movies, from the most rated to the least")
left.set_ylabel("number of ratings")
left.set_title("Most movies have few ratings, a few have hundreds")
tidy(left)
right.bar(stars.index, stars.to_numpy(), width=0.4, color=BLUE)
right.set_xticks(stars.index)
right.set_xlabel("stars given")
right.set_ylabel("number of ratings")
right.set_title("Whole stars far outnumber half stars")
tidy(right)
fig.tight_layout()
plt.show()

print(f"ratings of a movie: median {ratings_per_movie.median():.0f}, largest {ratings_per_movie.max()}; "
      f"{(ratings_per_movie == 1).sum():,} movies have one rating, {(ratings_per_movie >= 20).sum():,} have 20 or more")
print(f"ratings of a person: smallest {ratings_per_person.min()}, median {ratings_per_person.median():.0f}, largest {ratings_per_person.max():,}")

**The long tail.** Half of the movies have three ratings or fewer, and a few have more than three hundred. A column with three numbers says little about a movie, so this notebook keeps the **movies with 20 or more ratings**: about 1,300 of them, which hold two thirds of all the ratings. The matrix of people by kept movies has about 800,000 cells, which fits in memory many times over, with a rating in about one cell in twelve.

The next cell builds that matrix, `R`, with one row for each person and one column for each kept movie. A cell without a rating holds `NaN`, "not a number", the marker of a **missing value**. Beside it, `B` records where the ratings are: `True` where a cell of `R` holds a rating, `False` where it does not.

In [ ]:
MIN_RATINGS = 20
kept_ids = ratings_per_movie[ratings_per_movie >= MIN_RATINGS].index
kept = ratings[ratings.movieId.isin(kept_ids)]

table = kept.pivot(index="userId", columns="movieId", values="rating")      # people by movies; NaN where there is no rating
people = table.index.to_numpy()                                              # the id of the person in each row
movie_ids = table.columns.to_numpy()                                         # the id of the movie in each column
titles = np.array([short(t) for t in movies.set_index("movieId").loc[movie_ids, "title"]])
years = np.array([year_of(t) for t in movies.set_index("movieId").loc[movie_ids, "title"]])
title_counts = Counter(titles)
titles = np.array([f"{t} ({years[k]}; movieId {movie_ids[k]})" if title_counts[t] > 1 else t
                   for k, t in enumerate(titles)])                          # distinguish remakes instead of silently replacing one
assert len(set(titles)) == len(titles)
R = table.to_numpy()                                                         # the matrix of ratings, NaN where missing
B = ~np.isnan(R)                                                             # True where there is a rating

print(f"R has {R.shape[0]} rows (people) and {R.shape[1]:,} columns (movies): {R.size:,} cells, {B.sum():,} of them with a rating")
print(f"the kept movies hold {len(kept) / len(ratings) * 100:.0f}% of all the ratings")
print("the first 5 people and the first 8 movies:")
print(pd.DataFrame(R[:5, :8], index=people[:5], columns=titles[:8]))

<font color="#c00000" size="5"><b>Your turn.</b></font> In the next cell, compute `OBSERVED`, the fraction of the cells of `R` that hold a rating, from `R` or from `B`. `np.isnan(R)` is `True` where a value is missing; the mean of an array of `True` and `False` is the fraction of `True`. Then run the check.

In [ ]:
OBSERVED = ...      # Your turn: the fraction of the cells of R that hold a rating

In [ ]:
assert OBSERVED is not ..., "Replace the ... after OBSERVED, and run that cell again."
expected = B.mean()
if isinstance(OBSERVED, float) and np.isnan(OBSERVED):
    raise AssertionError("OBSERVED is NaN: the mean of R itself is missing when any cell is missing. Count the cells that hold a "
                         "rating instead, with B or with np.isnan.")
assert isinstance(OBSERVED, (int, float, np.floating)), "OBSERVED must be one number, a fraction between 0 and 1."
if abs(OBSERVED - (1 - expected)) < 1e-6:
    raise AssertionError("That is the fraction of the cells that are missing. OBSERVED is the fraction that hold a rating: "
                         "1 minus it, or the mean of B.")
assert abs(OBSERVED - expected) < 1e-6, "OBSERVED does not agree with R: B.mean(), or (~np.isnan(R)).mean()."
print(f"Correct: {OBSERVED * 100:.1f}% of the cells hold a rating, and {100 - OBSERVED * 100:.1f}% are missing.")

**What a missing value is.** A person who did not rate a movie did not give it zero stars: the cell is empty, not 0. The rest of this notebook fills the empty cells in two ways. In Part 4 it puts 0 in them, as they are; and it also subtracts from every rating the mean rating of the person who gave it, and then puts 0 in the empty cells. Answer in the next cell, in two or three sentences: what does each of the two choices say about a movie that a person did not rate? Which of the two is closer to "no opinion"?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 2: Features of a movie, as algebra</b></font>

A **feature** of a movie is one number describing it. The number of ratings, their mean and their spread come from its column of `R`; the year comes from `movies.csv`, and the number of tags from `tags.csv`. Operations on all the rating columns at once can be written as matrix operations.

The number of ratings of every movie is the sum of each column of `B`, and a column sum is a product with a vector of ones: `np.ones(610) @ B` multiplies every row of `B` by 1 and adds the rows. The mean rating of every movie is the sum of its ratings divided by their number; `np.nan_to_num(R)` turns the missing cells into 0, so that the column sums are the sums of the ratings. The next cell computes five features for every movie and puts them in a table `F`.

In [ ]:
ones = np.ones(len(people))
count = ones @ B                                            # the number of ratings of each movie: a column sum, as a product
assert np.array_equal(count, B.sum(axis=0))                 # the same as summing each column
mean = (ones @ np.nan_to_num(R)) / count                    # the mean rating of each movie
spread = np.nanstd(R, axis=0)                               # the standard deviation of the ratings of each movie, missing cells left out
tags = pd.read_csv(MOVIES + "tags.csv").groupby("movieId").size().reindex(movie_ids, fill_value=0).to_numpy()

F = pd.DataFrame({"title": titles, "count": count.astype(int), "mean": mean, "spread": spread, "year": years, "tags": tags})
print("the 8 most-rated movies:")
print(F.sort_values("count", ascending=False).head(8).to_string(index=False))
print()
print("the 8 movies with the highest mean rating:")
print(F.sort_values("mean", ascending=False).head(8).to_string(index=False))

**Mean rating by genre.** Each movie has one or more **genres**, listed in `movies.csv` with `|` between them. The next cell builds the **genre matrix** `G`: one row for each kept movie, one column for each genre, 1 where the movie has the genre and 0 where it does not. Then `G.T @ mean` adds up, for each genre, the mean ratings of the movies that have it, and `G.T @ ones_movies` counts those movies. Their ratio is the **mean of movie means**: each movie has equal weight, regardless of its number of ratings, and a movie with several genres contributes to each. This differs from pooling all individual ratings within a genre. In pandas, splitting the genre lists into separate rows and grouping by genre gives the same calculation; here it is two products.

In [ ]:
G = movies.set_index("movieId").loc[movie_ids, "genres"].str.get_dummies(sep="|")
G = G.loc[:, G.sum() > 0]                                   # a genre that no kept movie has would be a column of zeros
genre_names = list(G.columns)
G = G.to_numpy().astype(float)
print(f"G has {G.shape[0]:,} rows (movies) and {G.shape[1]} columns (genres)")
print("movies with each genre:", dict(zip(genre_names, G.sum(axis=0).astype(int))))

ones_movies = np.ones(len(movie_ids))
mean_per_genre = (G.T @ mean) / (G.T @ ones_movies)         # the sum of the means in each genre, divided by the number of movies
order = np.argsort(-mean_per_genre)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(range(len(order)), mean_per_genre[order], color=BLUE)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([genre_names[k] for k in order], rotation=60, ha="right")
ax.set_ylim(3.0, 4.2)
ax.set_ylabel("mean of the mean ratings")
ax.set_title("The mean rating of the movies of each genre, G.T @ mean / G.T @ ones")
tidy(ax)
fig.tight_layout()
plt.show()

<font color="#c00000" size="5"><b>Your turn.</b></font> Read the chart: in the next cell, type the genre with the highest mean rating as `HIGHEST_GENRE` and the genre with the lowest as `LOWEST_GENRE`, in quotes, spelled as on the chart. Then run the check.

In [ ]:
HIGHEST_GENRE = "..."      # Your turn: the genre whose movies have the highest mean rating
LOWEST_GENRE = "..."       # Your turn: the genre with the lowest

In [ ]:
for name, value in (("HIGHEST_GENRE", HIGHEST_GENRE), ("LOWEST_GENRE", LOWEST_GENRE)):
    assert value != "...", f"Replace the ... in {name} with a genre from the chart, and run that cell again."
    assert value in genre_names, f"{name} is '{value}', which is not one of the genres: spell it as on the chart, for example 'Sci-Fi'."
highest, lowest = genre_names[order[0]], genre_names[order[-1]]
if HIGHEST_GENRE == lowest and LOWEST_GENRE == highest:
    raise AssertionError("The two are swapped: the highest bar is the highest mean rating.")
assert HIGHEST_GENRE == highest, f"HIGHEST_GENRE is the genre of the tallest bar, at {mean_per_genre[order[0]]:.2f}."
assert LOWEST_GENRE == lowest, f"LOWEST_GENRE is the genre of the shortest bar, at {mean_per_genre[order[-1]]:.2f}."
print(f"Correct: {highest} is rated highest on average ({mean_per_genre[order[0]]:.2f}) and {lowest} lowest ({mean_per_genre[order[-1]]:.2f}).")

**Two features against each other.** The **correlation coefficient** of two features, `r`, is a number between -1 and 1 that says how well a straight line relates them. It is computed in three steps: **standardize** each feature (subtract its mean and divide by its standard deviation, as in meeting 10), multiply the two standardized columns entry by entry, and average the products. The average of products is a dot product divided by n, so `r` is one line of NumPy. The next cell draws two pairs of features and prints `r` for each, computed that way and by `np.corrcoef`, which must agree.

In [ ]:
def standardize(column):
    """The column with its mean subtracted and divided by its standard deviation: mean 0, standard deviation 1."""
    column = np.asarray(column, dtype=float)               # use the same convention for arrays and pandas Series
    scale = column.std(ddof=0)                             # divide by n, matching the dot-product formula below
    if not np.isfinite(column).all() or scale == 0:
        raise ValueError("Standardization needs finite values and a nonconstant column.")
    return (column - column.mean()) / scale


def correlation(x, y):
    """The correlation coefficient of two columns: the average product of the standardized columns, a dot product over n."""
    return standardize(x) @ standardize(y) / len(x)


log_count = np.log10(count)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.6))
left.scatter(spread, mean, s=8, color=BLUE, alpha=0.4, linewidths=0)
left.set_xlabel("spread: the standard deviation of the ratings of a movie")
left.set_ylabel("mean rating of the movie")
left.set_title(f"spread against mean: r = {correlation(spread, mean):+.2f}")
tidy(left)
right.scatter(log_count, mean, s=8, color=BLUE, alpha=0.4, linewidths=0)
right.set_xlabel("log10 of the number of ratings")
right.set_ylabel("mean rating of the movie")
right.set_title(f"number of ratings against mean: r = {correlation(log_count, mean):+.2f}")
tidy(right)
fig.tight_layout()
plt.show()

print(f"r of spread and mean: {correlation(spread, mean):+.3f} as a dot product, {np.corrcoef(spread, mean)[0, 1]:+.3f} by np.corrcoef")
print(f"r of year and mean: {correlation(years.astype(float), mean):+.3f}")

**Reading a scatter.** The left picture relates the spread of a movie's ratings to their mean. Describe it in two or three sentences in the next cell: what the sign of `r` says, and what the ceiling of five stars does to the spread of a movie whose mean is near 5.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 3: Correlation of features</b></font>

Part 2 computed `r` for one pair of columns. For every pair of the 19 genre columns at once, standardize every column of `G` into a matrix `Z`, and form **`Z.T @ Z / n`**: cell (i, j) of this product is the dot product of standardized columns i and j over n, which is their correlation. The result is the **correlation matrix**, 19 by 19, with 1 on its diagonal (a column with itself) and the same number at (i, j) and (j, i). The next cell computes it, checks it against `np.corrcoef`, draws it with blue for negative and orange for positive, and lists the strongest pairs.

In [ ]:
Z = (G - G.mean(axis=0)) / G.std(axis=0)                    # every genre column standardized
C = Z.T @ Z / len(Z)                                        # the correlation matrix of the 19 genres
assert np.allclose(C, np.corrcoef(G, rowvar=False)), "Z.T @ Z / n must be the correlation matrix"
print(f"Z is {Z.shape[0]:,} by {Z.shape[1]}; Z.T @ Z / n is {C.shape[0]} by {C.shape[1]}: {Z.shape[0]:,} multiplications for each of its {C.size} cells")


def draw_correlation(C, names, title):
    """The matrix as coloured cells: blue below 0, grey at 0, orange above; the number written where |r| is at least 0.2."""
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(C, cmap=DIVERGING, vmin=-0.65, vmax=0.65)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=60, ha="right")
    ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and abs(C[i, j]) >= 0.2:
                ax.text(j, i, f"{C[i, j]:+.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    fig.tight_layout()
    plt.show()


draw_correlation(C, genre_names, "The correlation matrix of the 19 genres, in the order of the columns of G")

pairs = sorted(((C[i, j], genre_names[i], genre_names[j]) for i in range(len(C)) for j in range(i + 1, len(C))))
print("the three most negative pairs:", [f"{a} and {b}: {r:+.2f}" for r, a, b in pairs[:3]])
print("the three most positive pairs:", [f"{a} and {b}: {r:+.2f}" for r, a, b in pairs[-3:][::-1]])

**Blocks.** The order of the genres is alphabetical, so the related ones are scattered. Put the genres that are correlated next to each other and the picture shows **blocks**: groups of genres that occur together more often than independence would predict.

<font color="#c00000" size="5"><b>Your turn.</b></font> Ask an AI assistant (Gemini in Colab, ChatGPT, Claude, or another) to write the function `blocks_order`, which finds such an order. State all of these requirements in your prompt:

- the function is named `blocks_order` and takes one argument, `C`, a square NumPy array of correlations;
- it treats `1 - C` as dissimilarities between the items; clip tiny negative round-off values to 0 and set the diagonal to 0, without changing `C`. Group the items by hierarchical clustering with scipy, `scipy.cluster.hierarchy.linkage` with the method `"average"` on the condensed distances (`scipy.spatial.distance.squareform` turns the square array into the condensed form; pass `checks=False`), with `optimal_ordering=True`;
- it returns the order of the items as `scipy.cluster.hierarchy.leaves_list` gives it: a NumPy array of the whole numbers 0 to n - 1, each once.

Paste your prompt into the next cell, and the function into the code cell after it. Then run the check, which measures how much correlation the order puts next to each other.

<font color="#c00000" size="5"><b>Your prompt.</b></font> Double-click this cell and replace this line with the prompt you gave to the AI assistant. If the first function did not pass the check, add one sentence on what you changed.

In [ ]:
# Your turn: paste the function blocks_order here, then run this cell.

In [ ]:
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

assert "blocks_order" in globals(), "Paste the function blocks_order into the cell above and run that cell."
distances = np.maximum(1 - C, 0)
np.fill_diagonal(distances, 0)
reference = leaves_list(linkage(squareform(distances, checks=False), "average", optimal_ordering=True))
neighbours = lambda order: sum(C[order[k], order[k + 1]] for k in range(len(order) - 1))   # positive correlation put next to each other

original_C = C.copy()
result = blocks_order(C)
assert np.array_equal(C, original_C), "blocks_order must not change the input correlation matrix."
result = np.asarray(result)
assert result.ndim == 1 and len(result) == len(C), f"blocks_order must return one order of the {len(C)} items, not {result.shape}."
assert np.issubdtype(result.dtype, np.integer), "blocks_order must return integer indices, not floats or strings."
assert sorted(result.tolist()) == list(range(len(C))), "blocks_order must return every item once: the whole numbers 0 to n - 1 in some order."
if neighbours(result) < 0.9 * neighbours(reference):
    raise AssertionError(f"The order puts {neighbours(result):.2f} of correlation next to each other; the clustering with average linkage and "
                         f"optimal ordering puts {neighbours(reference):.2f}. Is 1 - C the distance, and is optimal_ordering on?")
genre_order = result
print(f"Correct: the order puts {neighbours(result):.2f} of correlation next to each other, against {neighbours(np.arange(len(C))):.2f} "
      "for the alphabetical order.")
draw_correlation(C[np.ix_(genre_order, genre_order)], [genre_names[k] for k in genre_order],
                 "The same matrix, the genres ordered so that blocks appear")

**Reading the blocks.** In the next cell, name two blocks of the ordered picture (the genres in each) and one pair with a negative correlation, and say in a sentence what each means: which kinds of movies the data says go together, and which do not.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 4: Similarity of points: two products</b></font>

`B` has one row for each person and one column for each movie, so it can be multiplied two ways. With numeric 0/1 entries, **`B.T @ B`** is 1,297 by 1,297: cell (i, j) counts the people who rated both movie i and movie j. **`B @ B.T`** is 610 by 610: cell (i, j) counts the movies that both person i and person j rated. Our `B` currently has Boolean entries, so first convert it to numbers as `Bf = B.astype(float)`; NumPy's Boolean matrix product returns True/False, not counts. The next cell computes both numeric products, times them, and draws them with the movies and the people sorted by their number of ratings.

In [ ]:
Bf = B.astype(float)
start = time.perf_counter()
shared_raters = Bf.T @ Bf                                   # movies by movies: the people who rated both
time_movies = time.perf_counter() - start
start = time.perf_counter()
shared_movies = Bf @ Bf.T                                   # people by people: the movies both rated
time_people = time.perf_counter() - start
n_people, n_movies = B.shape
print(f"B.T @ B is {shared_raters.shape[0]:,} by {shared_raters.shape[1]:,}: {n_people * n_movies * n_movies / 1e9:.2f} billion multiply-adds, {time_movies:.2f} s")
print(f"B @ B.T is {shared_movies.shape[0]} by {shared_movies.shape[1]}: {n_movies * n_people * n_people / 1e9:.2f} billion multiply-adds, {time_people:.2f} s")

off = shared_raters - np.diag(np.diag(shared_raters))       # the diagonal is each movie's own count; leave it out of the search
i, j = np.unravel_index(np.argmax(off), off.shape)
print(f"the pair of movies with the most raters in common: {titles[i]} and {titles[j]}, {off[i, j]:.0f} people")
off = shared_movies - np.diag(np.diag(shared_movies))
i, j = np.unravel_index(np.argmax(off), off.shape)
print(f"the pair of people with the most movies in common: {people[i]} and {people[j]}, {off[i, j]:.0f} movies")

by_count = np.argsort(-count)
by_activity = np.argsort(-B.sum(axis=1))
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 5.4), gridspec_kw={"width_ratios": [n_movies / n_people, 1]})
left.imshow(np.log1p(shared_raters[np.ix_(by_count, by_count)]), cmap="Blues")
left.set_title(f"B.T @ B: {n_movies:,} movies, shared raters (log colour)", fontsize=10)
right.imshow(np.log1p(shared_movies[np.ix_(by_activity, by_activity)]), cmap="Blues")
right.set_title(f"B @ B.T: {n_people} people, shared movies (log colour)", fontsize=10)
for ax in (left, right):
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()

**From counts to similarity.** Shared raters count activity, and movies with many ratings tend to share more raters. Meeting 10 compared word vectors by the **cosine similarity**: divide each vector by its length, and take the dot product. Applied to the columns of a matrix, the cosine of columns i and j is cell (i, j) of `U.T @ U`, where `U` is the matrix with every column divided by its length: again one product.

Which matrix? Three choices, and they answer three different questions:

- the has-rated matrix `B`, 0 or 1: two movies are similar when the same people rated both;
- the ratings as given, with 0 in the missing cells, `np.nan_to_num(R)`: similar when the same people gave both high stars;
- the **user-centred** ratings, `Rc`: subtract from every rating the person's mean over the kept movies they rated, then put 0 in the missing cells. A rating above that mean is positive, below it negative, and a missing one is 0, "no opinion" in this representation. Two movies gain similarity when the same people rate both above their usual level **or both below it**; opposite deviations reduce similarity. Missing entries contribute nothing to the dot product. This measures agreement in deviations, not proof that both movies were liked, and the pattern of missing ratings still matters.

The next cell computes the three similarity matrices and lists, for *Toy Story*, the seven most similar movies under each.

In [ ]:
Rc = np.nan_to_num(R - np.nanmean(R, axis=1, keepdims=True))            # user-centred: each rating minus the person's mean; missing -> 0


def cosine_of_columns(M):
    """The matrix of cosine similarities of the columns of M: every column divided by its length, then U.T @ U."""
    lengths = np.linalg.norm(M, axis=0, keepdims=True)
    if np.any(lengths == 0):
        raise ValueError("Cosine similarity is undefined for a zero-length column.")
    U = M / lengths
    return U.T @ U


S_rated = cosine_of_columns(Bf)                               # from the has-rated matrix
S_given = cosine_of_columns(np.nan_to_num(R))                 # from the ratings as given
S_centred = cosine_of_columns(Rc)                             # from the user-centred ratings
similarities = {"has-rated B": S_rated, "ratings as given": S_given, "user-centred ratings": S_centred}

title_row = {t: k for k, t in enumerate(titles)}


def find(text):
    """The titles that contain the text, with their number of ratings."""
    return [(str(t), int(count[k])) for k, t in enumerate(titles) if text.lower() in t.lower()]


def nearest(title, S, k=7):
    """The k movies most similar to the movie with this title, under the similarity matrix S."""
    row = S[title_row[title]].copy()
    row[title_row[title]] = -9                                            # a movie is not its own neighbour
    return [(str(titles[q]), round(float(row[q]), 2)) for q in np.argsort(-row)[:k]]


for name, S in similarities.items():
    print(f"{name:>22s}: ", [t for t, s in nearest("Toy Story", S)])

<font color="#c00000" size="5"><b>Your turn.</b></font> Choose a movie you know. The next cell prints the titles that contain a piece of text, with their number of ratings, so that you can find the exact title: edit the text and run it. Then type the title, exactly as printed, as `MY_MOVIE` in the cell after it, and run the check, which prints the three lists for your movie.

In [ ]:
print(find("star wars"))        # Your turn: replace "star wars" with a piece of the title of a movie you know, and run the cell

In [ ]:
MY_MOVIE = "..."       # Your turn: the exact title, as printed by find, in quotes

In [ ]:
assert MY_MOVIE != "...", "Replace the ... in MY_MOVIE with a title, and run that cell again."
assert isinstance(MY_MOVIE, str), "MY_MOVIE must be a title in quotes."
if MY_MOVIE not in title_row:
    close = find(MY_MOVIE)
    hint = f" Did you mean {close[0][0]!r}?" if close else " Use find() to see the exact spelling; only movies with 20 or more ratings are here."
    raise AssertionError(f"'{MY_MOVIE}' is not one of the {len(titles):,} titles.{hint}")
print(f"{MY_MOVIE}: {int(count[title_row[MY_MOVIE]])} ratings, mean {mean[title_row[MY_MOVIE]]:.2f}")
for name, S in similarities.items():
    print(f"{name:>22s}: ", [t for t, s in nearest(MY_MOVIE, S)])

**Forty movies in one picture.** The next cell draws the similarity of the 40 most-rated movies under the has-rated matrix and under the user-centred ratings, with the movies ordered by `blocks_order` so that blocks appear. Under the has-rated matrix every cell is positive, because every pair of popular movies shares raters. Under the centred ratings the blocks are the series (*The Lord of the Rings*, *Star Wars*) and the kinds of film that the same people like more than usual, and the blue cells are pairs that the same people rate in opposite directions.

In [ ]:
top40 = by_count[:40]
order40 = blocks_order(S_centred[np.ix_(top40, top40)])
shown = top40[order40]
labels = [t if len(t) <= 30 else t[:29] + "…" for t in titles[shown]]     # long titles cut to 30 characters
fig, axes = plt.subplots(1, 2, figsize=(14, 7.6))
for ax, (name, S) in zip(axes, [("has-rated B", S_rated), ("user-centred ratings", S_centred)]):
    ax.imshow(S[np.ix_(shown, shown)], cmap=DIVERGING, vmin=-0.75, vmax=0.75)
    ax.set_xticks(range(40))
    ax.set_yticks(range(40))
    ax.set_xticklabels(labels, rotation=90, fontsize=6.5)
    ax.set_yticklabels(labels, fontsize=6.5)
    ax.tick_params(length=0)
    ax.set_title(f"cosine similarity of the 40 most-rated movies: {name}", fontsize=10)
    for side in ax.spines.values():
        side.set_visible(False)
fig.tight_layout()
plt.show()

**Three answers.** For your movie, which list best captures agreement in how people rate movies relative to their usual ratings? Why does the has-rated list measure activity rather than taste? Explain why even the centred list is not proof that people liked both movies: what happens when someone rates both below their own mean? Three or four sentences, with examples from your lists.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

**The same questions, asked of words.** The word vectors of meeting 10 are a matrix of the same kind, 50,000 words by 300 numbers, and the meeting asks some of this notebook's questions of it. The product comparing every word with every word has 2.5 billion cells: 10 GB in float32, or 20 GB in float64, for the output alone. This can exceed a Colab runtime's available memory. Meeting 10 needed only one word's neighbours, so it computed one matrix-vector product instead of storing the whole matrix.

## <font color="#1967d2"><b>Part 5: The axes of a cloud</b></font>

Split the 610 people at random into two halves, and give every movie two numbers: its mean rating from one half, and from the other. The movies become a **cloud** of 1,297 points in the plane, and the cloud leans on the diagonal, because the two halves largely agree.

Along which direction is the cloud most spread out? Centre the cloud (subtract the mean point), and **project** every point on a direction `u` of length 1: the projection is the dot product of the centred point with `u`, and all the projections at once are `Xc @ u`. The **variance** of the projections says how spread out the cloud is along `u`. The direction with the largest variance is the **first principal axis**; the direction at a right angle to it is the second. `np.linalg.svd` finds them: it returns the directions as the rows of its third output, and the spread along each as its **singular values**. The next cell builds the cloud and draws its axes.

In [ ]:
shuffled = np.random.default_rng(0).permutation(len(people))  # repeatable even when this cell is rerun
half_a, half_b = shuffled[: len(people) // 2], shuffled[len(people) // 2:]
mean_a = np.nanmean(R[half_a], axis=0)                      # the mean rating of each movie from the people of half A
mean_b = np.nanmean(R[half_b], axis=0)                      # and from half B
both = ~np.isnan(mean_a) & ~np.isnan(mean_b)                # a movie that no one of a half rated has no mean there
X = np.column_stack([mean_a[both], mean_b[both]])           # the cloud: one row for each movie, two coordinates

centre = X.mean(axis=0)
Xc = X - centre                                             # the centred cloud
_, singular_values, directions = np.linalg.svd(Xc, full_matrices=False)
axis_1, axis_2 = directions[0], directions[1]               # unit vectors: the first and the second principal axis

fig, ax = plt.subplots(figsize=(6.8, 6.8))
ax.scatter(X[:, 0], X[:, 1], s=9, color=BLUE, alpha=0.35, linewidths=0)
for u, s, colour, label in [(axis_1, singular_values[0], "#1A1A1A", "axis 1"), (axis_2, singular_values[1], ORANGE, "axis 2")]:
    d = u * s / np.sqrt(len(X)) * 2                         # drawn with length 2 standard deviations
    ax.annotate("", xy=centre + d, xytext=centre - d, arrowprops=dict(arrowstyle="<->", color=colour, linewidth=2.2))
    ax.annotate(label, centre + d, xytext=(6, -12), textcoords="offset points", color=colour)
ax.set_xlabel(f"mean rating from {len(half_a)} people (half A)")
ax.set_ylabel(f"mean rating from the other {len(half_b)} people (half B)")
ax.set_aspect("equal")
ax.set_title(f"{len(X):,} movies rated by two halves of the people: r = {correlation(X[:, 0], X[:, 1]):.2f}")
tidy(ax)
plt.show()

print(f"axis 1: direction ({axis_1[0]:.2f}, {axis_1[1]:.2f}), singular value {singular_values[0]:.1f}, variance of the projections {np.var(Xc @ axis_1):.3f}")
print(f"axis 2: direction ({axis_2[0]:.2f}, {axis_2[1]:.2f}), singular value {singular_values[1]:.1f}, variance of the projections {np.var(Xc @ axis_2):.3f}")
print(f"the ratio of the two axes: {singular_values[0] / singular_values[1]:.2f}")
print(f"for comparison, the variance of the projections on (1, 0), the horizontal axis: {np.var(Xc @ np.array([1.0, 0.0])):.3f}")

<font color="#c00000" size="5"><b>Your turn.</b></font> The diagonal direction is the unit vector `(1, 1) / sqrt(2)`. In the next cell, compute `VARIANCE_ALONG_DIAGONAL`, the variance of the projections of the centred cloud `Xc` on that direction, as `np.var(Xc @ u)`. Then run the check, and compare the number with the variance along axis 1.

In [ ]:
u = ...                               # Your turn: the unit vector along the diagonal, (1, 1) divided by sqrt(2), as a NumPy array
VARIANCE_ALONG_DIAGONAL = ...         # Your turn: the variance of the projections of Xc on u

In [ ]:
assert u is not ... and VARIANCE_ALONG_DIAGONAL is not ..., "Replace the two ... in the cell above, and run it again."
u = np.asarray(u, dtype=float)
assert u.shape == (2,), "u must be a vector of two numbers."
if abs(np.linalg.norm(u) - np.sqrt(2)) < 1e-9:
    raise AssertionError("u is (1, 1), of length sqrt(2): divide it by sqrt(2) so that its length is 1; the projections are then in the "
                         "units of the data.")
assert abs(np.linalg.norm(u) - 1) < 1e-6 and abs(u[0] - u[1]) < 1e-6, "u must be (1, 1) / sqrt(2): both entries equal, length 1."
expected = np.var(Xc @ u)
assert abs(VARIANCE_ALONG_DIAGONAL - expected) < 1e-6, "VARIANCE_ALONG_DIAGONAL must be np.var(Xc @ u)."
print(f"Correct: the variance along the diagonal is {expected:.3f}, against {np.var(Xc @ axis_1):.3f} along axis 1, "
      f"which the svd found at ({axis_1[0]:.3f}, {axis_1[1]:.3f}): the diagonal is almost, not exactly, the best direction.")

**In 19 dimensions.** The same question for the genre matrix: each movie is a point with 19 coordinates, its standardized genre columns, and the cloud has 19 axes. `np.linalg.eigh(C)` finds them from the correlation matrix of Part 3: its **eigenvalues** are the variances along the axes, and its **eigenvectors** are the axes, each a list of 19 weights, one for each genre. The axes are the **principal components** of the data, and finding them is **principal component analysis**, PCA. The next cell computes them, draws the genres by their weights in the first two components, and draws the 19 eigenvalues, largest first.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(C)                # eigh: for a symmetric matrix; eigenvalues in increasing order
eigenvalues, eigenvectors = eigenvalues[::-1], eigenvectors[:, ::-1]      # largest first
share = eigenvalues / eigenvalues.sum()                       # the fraction of the variance along each component
component_1, component_2 = eigenvectors[:, 0], eigenvectors[:, 1]
if component_1[genre_names.index("Children")] < 0:            # the sign of an eigenvector is arbitrary; fix it for reading
    component_1 = -component_1
if component_2[genre_names.index("Action")] < 0:
    component_2 = -component_2

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.4), gridspec_kw={"width_ratios": [1.1, 1]})
left.axhline(0, color="#dddddd", linewidth=1)
left.axvline(0, color="#dddddd", linewidth=1)
left.scatter(component_1, component_2, s=34, color=BLUE, edgecolors="white")
for k, name in enumerate(genre_names):
    offset = (4, 8) if name == "Animation" else (4, -9) if name == "Children" else (4, 3)
    left.annotate(name, (component_1[k], component_2[k]), xytext=offset, textcoords="offset points", fontsize=8.5)
left.set_xlabel(f"weight in component 1 ({share[0] * 100:.0f}% of the variance)")
left.set_ylabel(f"weight in component 2 ({share[1] * 100:.0f}%)")
left.set_title("The 19 genres by their weights in components 1 and 2")
tidy(left)
right.bar(range(1, 20), eigenvalues, color=BLUE)
right.set_xticks(range(1, 20))
right.set_xlabel("component")
right.set_ylabel("eigenvalue: the variance along the component")
right.set_title("The 19 eigenvalues of the correlation matrix")
tidy(right)
fig.tight_layout()
plt.show()

for name, component in (("component 1", component_1), ("component 2", component_2)):
    order = np.argsort(component)
    print(f"{name}: negative weights {[genre_names[k] for k in order[:4]]}, positive weights {[genre_names[k] for k in order[-4:][::-1]]}")
print(f"components needed for half of the variance: {np.searchsorted(np.cumsum(share), 0.5) + 1} of 19; for 90%: {np.searchsorted(np.cumsum(share), 0.9) + 1}")

**A map of movies (optional extension).** Each movie is a point with 610 coordinates, its column of `Rc`. To centre this cloud, subtract the mean **across movies for each user**, using `axis=1`. Those row means are already zero up to round-off because of user centring and zero-filling; we subtract them explicitly to show the PCA step. The next cell finds the two directions along which the 1,297 movies are most spread, and plots the 50 most-rated movies. The first direction moves with the mean rating; read the second from the names. The cell takes a few seconds. You may skip this extension and continue with the genre question below.

In [ ]:
Xr = Rc - Rc.mean(axis=1, keepdims=True)                     # centre each user coordinate across the movie points
U_people, s_ratings, _ = np.linalg.svd(Xr, full_matrices=False)
map_xy = Xr.T @ U_people[:, :2]                               # each movie's coordinates along the two longest axes
if correlation(map_xy[:, 0], mean) < 0:                       # the sign of an axis is arbitrary: let the first grow with the mean rating
    map_xy[:, 0] = -map_xy[:, 0]
if map_xy[title_row["Independence Day"], 1] < 0:              # and let the second put the blockbusters at the top
    map_xy[:, 1] = -map_xy[:, 1]

top50 = by_count[:50]
fig, ax = plt.subplots(figsize=(12, 7.5))
ax.axhline(0, color="#dddddd", linewidth=1)
ax.axvline(0, color="#dddddd", linewidth=1)
ax.scatter(map_xy[top50, 0], map_xy[top50, 1], s=36, color=BLUE, edgecolors="white")
labelled = set()
for coordinate in range(2):
    extremes = top50[np.argsort(map_xy[top50, coordinate])]
    labelled.update(extremes[:3])
    labelled.update(extremes[-3:])
for k in sorted(labelled):
    label = re.sub(r"^Star Wars: Episode [IVX]+ - ", "", titles[k])
    offset = (4, 10) if titles[k] == "The Mask" else (4, -8) if titles[k] == "Ace Ventura: Pet Detective" else (4, 3)
    align = "left"
    if map_xy[k, 0] > np.median(map_xy[top50, 0]):
        offset, align = (-4, offset[1]), "right"
    ax.annotate(label, map_xy[k], xytext=offset, ha=align, textcoords="offset points", fontsize=7.5)
ax.set_xlabel(f"first axis of the cloud of ratings (its correlation with the mean rating: {correlation(map_xy[:, 0], mean):.2f})")
ax.set_ylabel("second axis")
ax.set_title("The 50 most-rated movies on the first two axes (extreme points labelled)")
tidy(ax)
plt.show()
print(pd.DataFrame({"movie": titles[top50], "axis 1": map_xy[top50, 0], "axis 2": map_xy[top50, 1]}).round(2).to_string(index=False))

**Naming an axis.** From the weights printed under the picture of the genres, name component 1 in your own words: what kind of movie is at its negative end, and what kind at its positive end? Then: *Musical* sits far along it although only 55 of the 1,297 movies have it. What does standardizing a column with few ones do to those few movies? Three or four sentences in the next cell.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 6: Rank and condition, read from the eigenvalues</b></font>

The 19 eigenvalues are the **spectrum** of the correlation matrix. It describes the spread of the standardized data. The number of eigenvalues that are not zero is the **rank** of `Z`: the number of directions along which the data is spread at all. The ratio of the largest eigenvalue to the smallest is the condition number of the correlation matrix, and its square root is the **condition number** of `Z` itself, the number of meeting 13; `np.linalg.cond(Z)` computes it. And a zero eigenvalue means a direction with no spread: a column that is a combination of the other columns.

The next cell checks the relation between the eigenvalues and `np.linalg.cond`.

In [ ]:
print(f"the largest eigenvalue {eigenvalues[0]:.3f}, the smallest {eigenvalues[-1]:.3f}, their ratio {eigenvalues[0] / eigenvalues[-1]:.2f}")
print(f"the square root of the ratio {np.sqrt(eigenvalues[0] / eigenvalues[-1]):.3f}; np.linalg.cond(Z) = {np.linalg.cond(Z):.3f}")
print(f"the rank of Z: {np.linalg.matrix_rank(Z)} of {Z.shape[1]} columns")

<font color="#c00000" size="5"><b>Your turn.</b></font> The next cell adds a twentieth column to `G`, `Family`, which is the sum of the columns `Animation` and `Children`: 1 for a movie that has one of the two, 2 for a movie that has both. Before running it, predict in the cell after this one what the smallest of the 20 eigenvalues will be: type `PREDICTED_SMALLEST = 0` if you expect zero (up to round-off), or a positive number if you expect the new column to add a direction of its own. Then run the check, and then the cell that computes it.

In [ ]:
PREDICTED_SMALLEST = ...      # Your turn: 0, or a positive number

In [ ]:
assert PREDICTED_SMALLEST is not ..., "Replace the ... after PREDICTED_SMALLEST with your prediction, and run that cell again."
assert isinstance(PREDICTED_SMALLEST, (int, float)), "Type PREDICTED_SMALLEST as a number."
if PREDICTED_SMALLEST > 1e-6:
    raise AssertionError("A column that is the sum of two other columns adds no direction. Standardization preserves the linear "
                         "dependence (with rescaled coefficients), so one eigenvalue is zero. Predict 0.")
assert np.isfinite(PREDICTED_SMALLEST) and abs(PREDICTED_SMALLEST) <= 1e-6, "Predict a finite number near zero; a correlation matrix cannot have a negative eigenvalue except for tiny round-off."
print("Correct: a column that is a combination of other columns gives an eigenvalue of zero, up to round-off. Now run the next cell.")

In [ ]:
family = G[:, genre_names.index("Animation")] + G[:, genre_names.index("Children")]
G20 = np.column_stack([G, family])
Z20 = (G20 - G20.mean(axis=0)) / G20.std(axis=0)
C20 = Z20.T @ Z20 / len(Z20)
eigenvalues_20 = np.linalg.eigvalsh(C20)[::-1]
print(f"the 20 eigenvalues: {eigenvalues_20}")
print(f"the smallest: {eigenvalues_20[-1]:.1e}, zero up to round-off")
print("The exact condition numbers of Z20 and C20 are infinite: both have a dependent column.")
print(f"the numerical rank of Z20: {np.linalg.matrix_rank(Z20)} of 20 columns; floating-point np.linalg.cond(Z20) = {np.linalg.cond(Z20):.1e}")
print("A huge finite computed value reflects round-off, not a finite condition number for the exactly dependent matrix.")

noisy = family + 0.05 * np.random.default_rng(14).standard_normal(len(family))  # fixed noise; reruns agree
G20n = np.column_stack([G, noisy])
Z20n = (G20n - G20n.mean(axis=0)) / G20n.std(axis=0)
eigenvalues_20n = np.linalg.eigvalsh(Z20n.T @ Z20n / len(Z20n))[::-1]
print()
print(f"with noise on the Family column: the smallest eigenvalue {eigenvalues_20n[-1]:.4f}, the ratio {eigenvalues_20n[0] / eigenvalues_20n[-1]:.0f}, "
      f"the rank {np.linalg.matrix_rank(Z20n)}, np.linalg.cond(Z20n) = {np.linalg.cond(Z20n):.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(1, 21) - 0.2, np.abs(eigenvalues_20), width=0.38, color=BLUE, label="Family = Animation + Children")
ax.bar(np.arange(1, 21) + 0.2, eigenvalues_20n, width=0.38, color=ORANGE, label="the same, with a little noise")
ax.set_xticks(range(1, 21))
ax.set_xlabel("component")
ax.set_ylabel("eigenvalue")
ax.set_title("The spectrum with a twentieth column that is, or nearly is, a sum of two others")
ax.legend(frameon=False)
tidy(ax)
plt.show()

**Nearly a sum.** With the noise, the new column is no longer an exact sum of two others. In three or four sentences in the next cell: what happened to the smallest eigenvalue, to the rank and to the condition number, and what does the picture of the nearly parallel lines of meeting 13 say about solving a system whose matrix has such a column?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 7: Your claim</b></font>

For this exercise, make a **claim** about the movie data and support it with numbers and a plot. Then **check** how the result changes under a different preparation of the data. Some claims that the tables and pictures above suggest, to start from or to replace with your own:

- older movies in this data have higher mean ratings;
- the people who rate the most movies give lower ratings on average;
- the two halves of the people agree more on the movies with many ratings than on those with few;
- the movies most similar to a series film under the centred ratings are the other films of the series, under the has-rated matrix they are not;
- one genre has both a high mean rating and a small spread.

The check: redo the main number for movies with at least 50 ratings, or with the ratings as given instead of centred, or with the other half of the people, and say whether it moved. For the 50-rating check, keep the earlier cells as they are: use `many = count >= 50` in your claim cell and select matching entries, for example `years[many]` and `mean[many]` (or `F.loc[many]`). For a movie similarity matrix, select **both** axes with `S[np.ix_(many, many)]`, along with `titles[many]`. Merely assigning `MIN_RATINGS = 50` at the end does not rebuild any earlier arrays. State which preparation you changed and keep the others fixed.

Write the code in the next cell; you may ask an AI assistant for help with it. The cell must run from top to bottom after the cells above it. A check that weakens or overturns your claim is a useful result: report it honestly. Keep claims about this dataset; its selected users, missing ratings and popularity filter do not establish what all viewers prefer or what causes a rating.

In [ ]:
# Your turn: the code of your claim: the numbers, one picture, and the check under a different preparation of the data.

**Your claim, in words.** In the next cell: the claim in one sentence; the numbers behind it; what the picture shows; and what the check showed, in a sentence: did the claim survive?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## Upload your notebook

**Every student uploads their own notebook.** It is graded on the cells marked <font color="#c00000"><b>Your thinking</b></font> and <font color="#c00000"><b>Your prompt</b></font>: eight in all. Replace the placeholder text in each graded cell with your own response.

Before downloading, restart the runtime and run your completed cells from top to bottom. Fix any errors and keep your plots and numerical results in the saved notebook. You may leave the optional movie-map code unrun.

1. Open the **File** menu, choose **Download**, and then **Download .ipynb**.
2. Upload the file to [**Upload Your Class Work - Meeting 14**](https://njit.instructure.com/courses/67357/assignments/765524).